# Étape 5 – ETL : du Data Lake (S3) au Data Warehouse (RDS PostgreSQL)

**Objectif** : mettre à disposition des équipes d'analyse des données **propres et structurées** dans une base SQL,
contenant les mêmes données nettoyées que le Data Lake (exigence de l'énoncé).

**Compétences du référentiel** :
- C4 – nettoyer et organiser les données dans l'entrepôt en écrivant des processus **ETL** ;
- C1 – concevoir une architecture **Data Warehouse** (schéma, contraintes, sécurité).

| Phase | Ce qui est fait |
|---|---|
| **Extract** | Lecture des fichiers `processed/` **depuis S3** (et non depuis le disque local) : le Data Lake est la source de référence |
| **Transform** | Découpage en **3 tables normalisées** (destinations, hôtels, association), dédoublonnage, typage, contrôles d'intégrité |
| **Load** | Création du schéma (`sql/schema.sql`) puis chargement dans **une seule transaction** : tout ou rien |

**Modèle de données** :

```
destinations (1) ───< destination_hotels >─── (1) hotels
  city_id (PK)          city_id (PK, FK)            hotel_id (PK)
  météo, score          hotel_id (PK, FK)           nom, URL, GPS, note...
                        search_position, distance_km
```
Un hôtel pouvant être proposé pour **plusieurs destinations voisines** (ex. Colmar / Eguisheim), la relation est
**plusieurs-à-plusieurs** : une table d'association évite de dupliquer les informations de l'hôtel.
Une **vue** `v_kayak_enriched` restitue la version « à plat » identique au fichier enrichi de S3.

**Sécurité** : identifiants lus dans `.env` ; connexion **chiffrée** (SSL) ; base accessible uniquement depuis
l'adresse IP autorisée (security group) ; stockage chiffré.

## 1. Imports et configuration

In [1]:
import os
from io import BytesIO
from pathlib import Path

import boto3
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


def find_project_root(start: Path) -> Path:
    """Remonte l'arborescence jusqu'au dossier qui contient requirements.txt (racine du projet)."""
    for folder in [start, *start.parents]:
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("Racine du projet introuvable (requirements.txt absent)")


PROJECT_ROOT = find_project_root(Path.cwd())
SCHEMA_FILE = PROJECT_ROOT / "sql" / "schema.sql"
# override=True : le fichier .env fait foi, même si le kernel a déjà chargé une ancienne valeur
load_dotenv(PROJECT_ROOT / ".env", override=True)

REQUIRED = ["S3_BUCKET_NAME", "RDS_HOST", "RDS_PORT", "RDS_DB_NAME", "RDS_USER", "RDS_PASSWORD"]
missing = [name for name in REQUIRED if not os.getenv(name)]
assert not missing, f"Variables absentes du fichier .env : {missing}"
BUCKET = os.getenv("S3_BUCKET_NAME")
print("Configuration complète (valeurs non affichées)")

Configuration complète (valeurs non affichées)


## 2. Extract – lecture des données nettoyées depuis le Data Lake S3

In [2]:
s3 = boto3.client("s3")


def read_csv_from_s3(key: str) -> pd.DataFrame:
    """Lit un fichier CSV stocké dans le bucket S3 du projet."""
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    return pd.read_csv(BytesIO(body))


weather_summary = read_csv_from_s3("processed/weather_summary.csv")
hotels_s3 = read_csv_from_s3("processed/hotels.csv")
enriched_s3 = read_csv_from_s3("processed/kayak_enriched.csv")  # servira de référence pour la vérification finale

for name, df in [("weather_summary", weather_summary), ("hotels", hotels_s3), ("kayak_enriched", enriched_s3)]:
    print(f"{name:<16} : {df.shape[0]:>4} lignes × {df.shape[1]} colonnes")

weather_summary  :   35 lignes × 12 colonnes
hotels           :  700 lignes × 16 colonnes
kayak_enriched   :  700 lignes × 26 colonnes


## 3. Transform – construction des 3 tables

- **destinations** : une ligne par ville, colonnes renommées pour être explicites, dates typées ;
- **hotels** : une ligne par hôtel (**dédoublonnage** sur `hotel_id`), après avoir vérifié que les doublons sont
  bien identiques (sinon, conflit de données à signaler) ;
- **destination_hotels** : le lien ville ↔ hôtel, avec le rang Booking et la distance.

In [3]:
destinations = weather_summary.rename(columns={
    "period_start": "weather_period_start",
    "period_end": "weather_period_end",
})[[
    "city_id", "city_name", "latitude", "longitude", "weather_period_start", "weather_period_end",
    "total_rain_mm", "avg_pop", "avg_temp_day", "rainy_days", "weather_score", "weather_rank",
]]
destinations["weather_period_start"] = pd.to_datetime(destinations["weather_period_start"]).dt.date
destinations["weather_period_end"] = pd.to_datetime(destinations["weather_period_end"]).dt.date

HOTEL_COLUMNS = [
    "hotel_id", "hotel_name", "hotel_url", "accommodation_type_id", "latitude", "longitude", "address",
    "hotel_city", "review_score", "review_count", "star_rating", "description", "collected_at",
]
# Un hôtel présent pour deux destinations doit avoir exactement les mêmes informations
conflicts = hotels_s3[HOTEL_COLUMNS].drop_duplicates().duplicated(subset="hotel_id").sum()
assert conflicts == 0, f"{conflicts} hôtels ont des informations différentes selon la destination"

hotels = hotels_s3[HOTEL_COLUMNS].drop_duplicates(subset="hotel_id").copy()
hotels["star_rating"] = hotels["star_rating"].astype("Int64")           # entier pouvant être vide (non classé)
hotels["collected_at"] = pd.to_datetime(hotels["collected_at"], utc=True)

destination_hotels = hotels_s3[["city_id", "hotel_id", "search_position", "distance_km"]].copy()

print(f"destinations       : {len(destinations)} lignes")
print(f"hotels             : {len(hotels)} lignes ({len(hotels_s3) - len(hotels)} doublons retirés)")
print(f"destination_hotels : {len(destination_hotels)} lignes")

destinations       : 35 lignes
hotels             : 693 lignes (7 doublons retirés)
destination_hotels : 700 lignes


In [4]:
# Contrôles d'intégrité avant chargement (la base les vérifiera aussi grâce aux contraintes)
assert destinations["city_id"].is_unique, "city_id doit être unique"
assert hotels["hotel_id"].is_unique, "hotel_id doit être unique"
assert not destination_hotels.duplicated(subset=["city_id", "hotel_id"]).any(), "lien ville-hôtel en double"
assert destination_hotels["city_id"].isin(destinations["city_id"]).all(), "ville inconnue dans les liens"
assert destination_hotels["hotel_id"].isin(hotels["hotel_id"]).all(), "hôtel inconnu dans les liens"
print("Contrôles d'intégrité : OK")

Contrôles d'intégrité : OK


## 4. Load – création du schéma et chargement dans RDS

- La connexion utilise `URL.create` : le mot de passe est transmis proprement, même s'il contient des caractères
  spéciaux (`@`, `/`, `%`…), et n'apparaît jamais dans le code ;
- `sslmode=require` : les échanges entre le PC et la base sont **chiffrés** ;
- `engine.begin()` ouvre une **transaction** : si une seule étape échoue, rien n'est modifié dans la base ;
- le schéma est entièrement recréé à chaque exécution (**rechargement complet**), ce qui rend l'ETL
  **rejouable** sans créer de doublons – adapté à ce volume de données.

In [5]:
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("RDS_USER"),
    password=os.getenv("RDS_PASSWORD"),
    host=os.getenv("RDS_HOST"),
    port=int(os.getenv("RDS_PORT")),
    database=os.getenv("RDS_DB_NAME"),
)
engine = create_engine(db_url, connect_args={"sslmode": os.getenv("RDS_SSLMODE", "require")})

with engine.connect() as connection:
    version = connection.execute(text("SELECT version()")).scalar()
    ssl = connection.execute(text("SHOW ssl")).scalar()
print("Connexion OK :", version.split(",")[0], "| SSL activé côté serveur :", ssl)

Connexion OK : PostgreSQL 18.3 on aarch64-unknown-linux-gnu | SSL activé côté serveur : on


In [6]:
with engine.begin() as connection:  # transaction : tout ou rien
    connection.exec_driver_sql(SCHEMA_FILE.read_text(encoding="utf-8"))
    destinations.to_sql("destinations", connection, if_exists="append", index=False)
    hotels.to_sql("hotels", connection, if_exists="append", index=False)
    destination_hotels.to_sql("destination_hotels", connection, if_exists="append", index=False)
print("Chargement terminé")

Chargement terminé


## 5. Vérifications après chargement

In [7]:
with engine.connect() as connection:
    counts = {
        table: connection.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        for table in ["destinations", "hotels", "destination_hotels", "v_kayak_enriched"]
    }
counts

{'destinations': 35,
 'hotels': 693,
 'destination_hotels': 700,
 'v_kayak_enriched': 700}

### La base restitue-t-elle les mêmes données que S3 ?

On compare la vue `v_kayak_enriched` au fichier `processed/kayak_enriched.csv` du Data Lake,
ligne à ligne (clé : ville + hôtel) et colonne par colonne.

In [8]:
view = pd.read_sql("SELECT * FROM v_kayak_enriched", engine)

keys = ["city_id", "hotel_id"]
left = enriched_s3.sort_values(keys).reset_index(drop=True)
right = view[left.columns].sort_values(keys).reset_index(drop=True)

# Mise au même format des colonnes de dates avant comparaison
for column in ["weather_period_start", "weather_period_end"]:
    left[column] = pd.to_datetime(left[column]).dt.date
    right[column] = pd.to_datetime(right[column]).dt.date
left["hotels_collected_at"] = pd.to_datetime(left["hotels_collected_at"], utc=True)
right["hotels_collected_at"] = pd.to_datetime(right["hotels_collected_at"], utc=True)

differences = []
for column in left.columns:
    a, b = left[column], right[column]
    if pd.api.types.is_numeric_dtype(a) or pd.api.types.is_numeric_dtype(b):
        a, b = pd.to_numeric(a), pd.to_numeric(b)
        same = ((a - b).abs() < 1e-6) | (a.isna() & b.isna())
    else:
        same = (a.astype(str) == b.astype(str)) | (a.isna() & b.isna())
    if not same.all():
        differences.append((column, int((~same).sum())))

print(f"S3 : {left.shape} | RDS : {right.shape}")
print("Données identiques entre S3 et RDS" if not differences else f"Différences : {differences}")

S3 : (700, 26) | RDS : (700, 26)
Données identiques entre S3 et RDS


## 6. Exemples de requêtes pour les équipes d'analyse

In [9]:
# Les 5 meilleures destinations selon la météo des 7 prochains jours
pd.read_sql("""
    SELECT weather_rank, city_name, weather_score, total_rain_mm, avg_temp_day,
           weather_period_start, weather_period_end
    FROM destinations
    ORDER BY weather_rank
    LIMIT 5
""", engine)

,weather_rank,city_name,weather_score,total_rain_mm,avg_temp_day,weather_period_start,weather_period_end
0,1,Collioure,0.933,1.0,16.8,2026-11-05,2026-11-11
1,2,Bayonne,0.914,5.6,17.0,2026-11-05,2026-11-11
2,3,Carcassonne,0.894,3.0,15.6,2026-11-05,2026-11-11
3,4,Biarritz,0.890,5.6,16.5,2026-11-05,2026-11-11
4,5,Toulouse,0.882,3.7,15.4,2026-11-05,2026-11-11


In [10]:
# Les 20 hôtels les mieux notés des 5 meilleures destinations (à 20 km maximum de la destination)
pd.read_sql("""
    SELECT d.city_name, h.hotel_name, h.review_score, h.review_count, h.star_rating, dh.distance_km
    FROM destination_hotels AS dh
    JOIN destinations AS d ON d.city_id = dh.city_id
    JOIN hotels       AS h ON h.hotel_id = dh.hotel_id
    WHERE d.weather_rank <= 5
      AND dh.distance_km <= 20
      AND h.review_score IS NOT NULL
    ORDER BY h.review_score DESC, h.review_count DESC
    LIMIT 20
""", engine)

,city_name,hotel_name,review_score,review_count,star_rating,distance_km
0,Biarritz,Maison Atalaye - Hôtel entièrement rénové,9.5,76,3.0,1.6
1,Biarritz,Hôtel Au Bon Coin,9.4,1541,3.0,1.2
2,Biarritz,"Hôtel du Palais Biarritz, in The Unbound Colle...",9.4,766,5.0,1.7
3,Bayonne,"Hôtel du Palais Biarritz, in The Unbound Colle...",9.4,766,5.0,6.7
4,Toulouse,Boutique Hotel SOCLO,9.4,422,4.0,0.5
5,Biarritz,Maison Resaurie - Hôtel,9.4,11,2.0,2.1
6,Biarritz,Hôtel de La Plage,9.3,761,4.0,1.7
7,Collioure,Les Roches Brunes,9.3,653,4.0,0.6
8,Carcassonne,Hôtel La Bastide Saint Martin,9.3,555,3.0,4.2
9,Toulouse,Alfred Hotels Toulouse Centre Wilson - Hôtel R...,9.3,492,3.0,0.5


In [11]:
# Hôtels proposés pour plusieurs destinations (intérêt de la table d'association)
pd.read_sql("""
    SELECT h.hotel_name, STRING_AGG(d.city_name, ' / ' ORDER BY d.city_name) AS destinations
    FROM destination_hotels AS dh
    JOIN hotels       AS h ON h.hotel_id = dh.hotel_id
    JOIN destinations AS d ON d.city_id = dh.city_id
    GROUP BY h.hotel_name
    HAVING COUNT(*) > 1
""", engine)

,hotel_name,destinations
0,Colmar Hotel,Colmar / Eguisheim
1,"Hôtel du Palais Biarritz, in The Unbound Colle...",Bayonne / Biarritz
2,Hôtel Le Bellevue,Ariege / Ariege
3,Hotel Le Colombier,Colmar / Eguisheim
4,James Boutique Hôtel Colmar centre,Colmar / Eguisheim
5,L'Esquisse Hotel & Spa Colmar - MGallery Colle...,Colmar / Eguisheim
6,Novotel Suites Colmar Centre,Colmar / Eguisheim
7,Relais & Châteaux La Maison Des Têtes,Colmar / Eguisheim
